# Fine-tuning BERT for customer intent classification

This notebook fine-tunes a BERT model to classify retail customer messages as `venda` or `suporte`.

In [ ]:
!pip install transformers datasets evaluate accelerate huggingface_hub

In [ ]:
# This BERT solution does not use the OpenAI API.
OPENAI_API_KEY = "Insira sua chave da OPENAI aqui"

In [ ]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)

## Load the JSONL datasets

In Google Colab, upload `treino.jsonl` and `teste.jsonl` to `/content/` before running this cell.

In [ ]:
DATA_DIR = "/content"

datasets = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/treino.jsonl",
        "test": f"{DATA_DIR}/teste.jsonl",
    },
)

datasets

## Configure labels and tokenizer

The Portuguese BERT checkpoint is used because the customer messages are written in Portuguese.

In [ ]:
checkpoint = "neuralmind/bert-base-portuguese-cased"
label_to_id = {"suporte": 0, "venda": 1}
id_to_label = {0: "suporte", 1: "venda"}

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["prompt"], truncation=True)

def transform_labels(examples):
    return {"labels": [label_to_id[label] for label in examples["completion"]]}

tokenized_datasets = datasets.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.map(transform_labels, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["prompt", "completion"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Configure the model and training

The configuration follows the `Trainer` workflow used in class.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
    id2label=id_to_label,
    label2id=label_to_id,
)

training_args = TrainingArguments(
    output_dir="./bert-customer-intent",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)

In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Train, evaluate, and save the model

In [ ]:
trainer.train()
evaluation = trainer.evaluate()
print(evaluation)

trainer.save_model("./bert-customer-intent")
tokenizer.save_pretrained("./bert-customer-intent")

## Test a new message

In [ ]:
classifier = pipeline(
    "text-classification",
    model="./bert-customer-intent",
    tokenizer="./bert-customer-intent",
)

classifier("Quero comprar uma geladeira nova para minha casa.")